In [21]:
#| default_exp trigonometry

In [22]:
#| export

import jax
import jax.numpy as jnp

In [23]:
#| export

from jaxtyping import Float

In [24]:

#| hide

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_debug_nans", False)

In [25]:
#| hide

import jaxtyping


In [26]:

#| hide

%load_ext jaxtyping
%jaxtyping.typechecker beartype.beartype

The jaxtyping extension is already loaded. To reload it, use:
  %reload_ext jaxtyping


# Trigonometry

This module provides basic trigonometric, vector, and linear algebra functions for triangular meshes. These are scalar or single-triangle operations designed to be vectorized across meshes using `jax.vmap`.

::: {.callout-note collapse="true"}
### Coding style notes

Throughout, we provide type signatures for all functions using [jaxtyping](https://docs.kidger.site/jaxtyping). We use `jnp` (= `jax.numpy`) instead of `numpy`, and follow JAX's functional programming paradigm (see [JAX — the sharp bits](https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html)).


**Safe division.** To avoid division by zero in degenerate configurations (e.g. zero-area triangles), all divisors are clamped via `jnp.clip(x, 1e-12)`. The threshold `1e-12` is chosen to be well below any physically meaningful scale while staying above float64 precision limits (~`1e-16`).
:::

### Triangle areas and circumcenters

In [ ]:
#| export

def get_circumcenter(a: Float[jax.Array, " dim"],
                     b: Float[jax.Array, " dim"],
                     c: Float[jax.Array, " dim"]) -> Float[jax.Array, " dim"]:
    """Circumcenter of triangle with vertices a, b, c via barycentric coordinates.

    Parameters
    ----------
    a, b, c : Float[Array, "dim"]
        Triangle vertex positions.

    Returns
    -------
    Float[Array, "dim"]
        Circumcenter coordinates.
    """
    la, lb, lc = jnp.linalg.norm(b - c), jnp.linalg.norm(c - a), jnp.linalg.norm(a - b)
    ba = la**2 * (lb**2 + lc**2 - la**2)
    bb = lb**2 * (lc**2 + la**2 - lb**2)
    bc = lc**2 * (la**2 + lb**2 - lc**2)
    return (ba * a + bb * b + bc * c) / jnp.clip(ba + bb + bc, 1e-12)


def get_oriented_triangle_area(a: Float[jax.Array, " dim"],
                               b: Float[jax.Array, " dim"],
                               c: Float[jax.Array, " dim"]) -> Float[jax.Array, "*"]:
    """Signed area of triangle with vertices a, b, c.

    Parameters
    ----------
    a, b, c : Float[Array, "dim"]
        Triangle vertex positions (dim = 2 or 3).

    Returns
    -------
    Float[Array, "*"]
        Scalar (dim=2) or area-weighted normal vector (dim=3).
    """
    return 0.5 * jnp.cross(b - a, c - a)


def get_triangle_area(a: Float[jax.Array, " dim"],
                      b: Float[jax.Array, " dim"],
                      c: Float[jax.Array, " dim"]) -> Float[jax.Array, ""]:
    """Unsigned area of triangle with vertices a, b, c.

    Parameters
    ----------
    a, b, c : Float[Array, "dim"]
        Triangle vertex positions (dim = 2 or 3).

    Returns
    -------
    Float[Array, ""]
        Triangle area.
    """
    return 0.5 * jnp.linalg.norm(jnp.cross(b - a, c - a))


def get_polygon_area(pts: Float[jax.Array, "n_vertices 2"]) -> Float[jax.Array, ""]:
    """Signed area of a 2D simple polygon (shoelace formula).

    Positive for counter-clockwise vertex ordering.

    Parameters
    ----------
    pts : Float[Array, "n_vertices 2"]
        Ordered polygon vertices.

    Returns
    -------
    Float[Array, ""]
        Signed area.
    """
    return jnp.sum(pts[:, 0] * jnp.roll(pts[:, 1], 1)
                   - jnp.roll(pts[:, 0], 1) * pts[:, 1]) / 2

### Vector operations

In [28]:
#| export

def project_on_vector(a: Float[jax.Array, " dim"], b: Float[jax.Array, " dim"]
                      ) -> Float[jax.Array, " dim"]:
    """Project vector a onto vector b.

    Parameters
    ----------
    a : Float[Array, "dim"]
        Vector to project.
    b : Float[Array, "dim"]
        Target direction.

    Returns
    -------
    Float[Array, "dim"]
        Projection of a onto b.
    """
    return jnp.dot(a, b) / jnp.clip(jnp.dot(b, b), 1e-12) * b


def project_out_vector(a: Float[jax.Array, " dim"], b: Float[jax.Array, " dim"]
                       ) -> Float[jax.Array, " dim"]:
    """Project vector a onto the plane orthogonal to b.

    Parameters
    ----------
    a : Float[Array, "dim"]
        Input vector.
    b : Float[Array, "dim"]
        Normal direction to project out.

    Returns
    -------
    Float[Array, "dim"]
        Component of a orthogonal to b.
    """
    return a - jnp.dot(a, b) / jnp.clip(jnp.dot(b, b), 1e-12) * b


def get_projector(normal: Float[jax.Array, " dim"]) -> Float[jax.Array, "dim dim"]:
    """Tangent-plane projector $P = I - \\hat{n} \\otimes \\hat{n}$.

    Parameters
    ----------
    normal : Float[Array, "dim"]
        Normal vector (need not be unit length).

    Returns
    -------
    Float[Array, "dim dim"]
        Projection matrix onto the plane orthogonal to normal.
    """
    normal = normal / jnp.clip(jnp.linalg.norm(normal), 1e-12)
    return jnp.eye(normal.shape[0]) - jnp.outer(normal, normal)


def get_signed_angle_between_vectors(a: Float[jax.Array, "2"], b: Float[jax.Array, "2"]
                                     ) -> Float[jax.Array, ""]:
    """Signed angle from 2D vector a to b (CCW positive).

    Parameters
    ----------
    a, b : Float[Array, "2"]
        2D vectors.

    Returns
    -------
    Float[Array, ""]
        Signed angle in radians, in $(-\\pi, \\pi]$.
    """
    return jnp.atan2(jnp.cross(a, b), jnp.dot(a, b))


def get_angle_between_vectors(a: Float[jax.Array, " dim"], b: Float[jax.Array, " dim"]
                              ) -> Float[jax.Array, ""]:
    """Unsigned angle between vectors a and b.

    Parameters
    ----------
    a, b : Float[Array, "dim"]
        Input vectors.

    Returns
    -------
    Float[Array, ""]
        Angle in radians, in $[0, \\pi]$.
    """
    return jnp.atan2(jnp.linalg.norm(jnp.cross(a, b)), jnp.dot(a, b))


def get_cot_between_vectors(a: Float[jax.Array, " dim"], b: Float[jax.Array, " dim"]
                            ) -> Float[jax.Array, ""]:
    """Cotangent of the angle between vectors a and b.

    Parameters
    ----------
    a, b : Float[Array, "dim"]
        Input vectors.

    Returns
    -------
    Float[Array, ""]
        Cotangent value.
    """
    return jnp.dot(a, b) / jnp.clip(jnp.linalg.norm(jnp.cross(a, b)), 1e-12)

In [29]:
#| export

def get_tetrahedron_volume(a: Float[jax.Array, "3"], b: Float[jax.Array, "3"],
                           c: Float[jax.Array, "3"]) -> Float[jax.Array, ""]:
    """Signed volume of tetrahedron with edge vectors a, b, c from a common vertex.

    Parameters
    ----------
    a, b, c : Float[Array, "3"]
        Edge vectors from a shared origin vertex.

    Returns
    -------
    Float[Array, ""]
        Signed volume (positive if a, b, c form a right-handed frame).
    """
    return jnp.linalg.vecdot(a, jnp.cross(b, c)) / 6

In [30]:
#| hide
get_circumcenter(jnp.array([0.,0.]), jnp.array([0.,1.]),  jnp.array([1.,0.]))

Array([0.5, 0.5], dtype=float64)

In [31]:
#| hide
get_signed_angle_between_vectors(jnp.array([1.,0.]), jnp.array([0.5,0.5])) * (180/jnp.pi)

Array(45., dtype=float64)

In [32]:
#| hide
get_polygon_area(jnp.array([[0.,0.], [0.,1.], [1.,0.]])), get_triangle_area(*jnp.array([[0.,0.], [0.,1.], [1.,0.]]))

(Array(0.5, dtype=float64), Array(0.5, dtype=float64))

In [33]:
#| hide
get_circumcenter(jnp.array([1.,0.]), jnp.array([1.,0.]),  jnp.array([0.,1.]))

Array([0., 0.], dtype=float64)

In [34]:
#| hide
get_circumcenter(jnp.array([1.,0.]), jnp.array([2.,0.]),  jnp.array([1.,0.]))

Array([0., 0.], dtype=float64)

### Intrinsic geometry

Many mesh quantities (angles, triangle areas, ...) can be computed purely intrinsically from edge lengths $\ell_{ij}$ without explicit reference to the mesh vertex coordinates in 3d.

In [ ]:
#| export

def get_triangle_area_from_lengths(la: Float[jax.Array, ""],
                                   lb: Float[jax.Array, ""],
                                   lc: Float[jax.Array, ""]) -> Float[jax.Array, ""]:
    """Triangle area from side lengths using Heron's formula.

    Parameters
    ----------
    la : Float[Array, ""]
        Length of side opposite vertex a (i.e. edge bc).
    lb : Float[Array, ""]
        Length of side opposite vertex b (i.e. edge ca).
    lc : Float[Array, ""]
        Length of side opposite vertex c (i.e. edge ab).

    Returns
    -------
    Float[Array, ""]
        Area of the triangle.
    """
    s = (la + lb + lc) / 2
    return jnp.sqrt(jnp.clip(s * (s - la) * (s - lb) * (s - lc), 0.0))


def get_angles_from_lengths(la: Float[jax.Array, ""],
                            lb: Float[jax.Array, ""],
                            lc: Float[jax.Array, ""]) -> Float[jax.Array, "3"]:
    """Interior angles from side lengths using the law of cosines.

    Parameters
    ----------
    la : Float[Array, ""]
        Length of side opposite vertex a (i.e. edge bc).
    lb : Float[Array, ""]
        Length of side opposite vertex b (i.e. edge ca).
    lc : Float[Array, ""]
        Length of side opposite vertex c (i.e. edge ab).

    Returns
    -------
    Float[Array, "3"]
        Angles [alpha, beta, gamma] at vertices a, b, c respectively.
    """
    l2 = jnp.stack([la**2, lb**2, lc**2])               # [a², b², c²]
    l2_opp = jnp.roll(l2, -1) + jnp.roll(l2, -2) - l2   # [b²+c²-a², c²+a²-b², a²+b²-c²]
    denom = 2 * jnp.roll(jnp.stack([la, lb, lc]), -1) * jnp.roll(jnp.stack([la, lb, lc]), -2)
    cos_angles = jnp.clip(l2_opp / jnp.clip(denom, 1e-12), -1.0, 1.0)
    return jnp.arccos(cos_angles)


def get_cotangents_from_lengths(la: Float[jax.Array, ""],
                                lb: Float[jax.Array, ""],
                                lc: Float[jax.Array, ""]) -> Float[jax.Array, "3"]:
    """Cotangents of interior angles from side lengths.

    Uses $\\cot(\\alpha) = (b^2 + c^2 - a^2) / (4 \\cdot \\text{area})$.

    Parameters
    ----------
    la : Float[Array, ""]
        Length of side opposite vertex a (i.e. edge bc).
    lb : Float[Array, ""]
        Length of side opposite vertex b (i.e. edge ca).
    lc : Float[Array, ""]
        Length of side opposite vertex c (i.e. edge ab).

    Returns
    -------
    Float[Array, "3"]
        Cotangents [cot_alpha, cot_beta, cot_gamma] at vertices a, b, c.
    """
    l2 = jnp.stack([la**2, lb**2, lc**2])
    l2_opp = jnp.roll(l2, -1) + jnp.roll(l2, -2) - l2
    area4 = jnp.clip(4 * get_triangle_area_from_lengths(la, lb, lc), 1e-12)
    return l2_opp / area4


def get_circumcenter_from_lengths(la: Float[jax.Array, ""],
                                  lb: Float[jax.Array, ""],
                                  lc: Float[jax.Array, ""]) -> Float[jax.Array, "3"]:
    """Circumcenter in barycentric coordinates from edge lengths.

    To recover Cartesian coordinates: $u = \\lambda_a \\, a + \\lambda_b \\, b + \\lambda_c \\, c$.

    Parameters
    ----------
    la : Float[Array, ""]
        Length of side opposite vertex a (i.e. edge bc).
    lb : Float[Array, ""]
        Length of side opposite vertex b (i.e. edge ca).
    lc : Float[Array, ""]
        Length of side opposite vertex c (i.e. edge ab).

    Returns
    -------
    Float[Array, "3"]
        Normalized barycentric coordinates [lambda_a, lambda_b, lambda_c].
    """
    ba = la**2 * (lb**2 + lc**2 - la**2)
    bb = lb**2 * (lc**2 + la**2 - lb**2)
    bc = lc**2 * (la**2 + lb**2 - lc**2)
    return jnp.array([ba, bb, bc]) / jnp.clip(ba + bb + bc, 1e-12)

In [36]:
# Test intrinsic functions against extrinsic (vertex-based) implementations
triangles = [
    jnp.array([[0., 0.], [1., 0.], [0., 1.]]),
    jnp.array([[0., 0.], [3., 0.], [1.5, 2.]]),
    jnp.array([[0., 0., 0.], [1., 0., 0.], [0., 1., 1.]]),
    jnp.array([[1., 2.], [4., 6.], [7., 1.]]),]

for tri in triangles:
    a, b, c = tri
    la, lb, lc = jnp.linalg.norm(b - c), jnp.linalg.norm(c - a), jnp.linalg.norm(a - b)

    # Area
    assert jnp.allclose(get_triangle_area(a, b, c),
                         get_triangle_area_from_lengths(la, lb, lc), atol=1e-10)
    # Angles
    angles_ext = jnp.stack([get_angle_between_vectors(b - a, c - a),
                            get_angle_between_vectors(a - b, c - b),
                            get_angle_between_vectors(a - c, b - c)])
    assert jnp.allclose(angles_ext, get_angles_from_lengths(la, lb, lc), atol=1e-10)

    # Cotangents
    cots_ext = jnp.stack([get_cot_between_vectors(b - a, c - a),
                          get_cot_between_vectors(a - b, c - b),
                          get_cot_between_vectors(a - c, b - c)])
    assert jnp.allclose(cots_ext, get_cotangents_from_lengths(la, lb, lc), atol=1e-10)

    # Circumcenter
    cc_ext = get_circumcenter(a, b, c)
    bary = get_circumcenter_from_lengths(la, lb, lc)
    cc_int = bary[0] * a + bary[1] * b + bary[2] * c
    assert jnp.allclose(cc_ext, cc_int, atol=1e-10)

### Rotation matrices and normals

In [38]:
#| export

def get_rot_mat(theta: float) -> Float[jax.Array, "2 2"]:
    """2D counter-clockwise rotation matrix for angle theta.

    Parameters
    ----------
    theta : float
        Rotation angle in radians.

    Returns
    -------
    Float[Array, "2 2"]
        Rotation matrix.
    """
    return jnp.array([[jnp.cos(theta), -jnp.sin(theta)],
                      [jnp.sin(theta), jnp.cos(theta)]])


def get_perp_2d(x: Float[jax.Array, "... 2"]) -> Float[jax.Array, "... 2"]:
    """90-degree clockwise rotation of a 2D vector: $(x, y) \\to (y, -x)$.

    Parameters
    ----------
    x : Float[Array, "... 2"]
        Input vector(s).

    Returns
    -------
    Float[Array, "... 2"]
        Perpendicular vector(s).
    """
    return jnp.stack([x[..., 1], -x[..., 0]], axis=-1)


def get_triangle_normal(a: Float[jax.Array, "3"],
                        b: Float[jax.Array, "3"],
                        c: Float[jax.Array, "3"]) -> Float[jax.Array, "3"]:
    """Unit normal of triangle abc (right-hand rule on a -> b -> c).

    Parameters
    ----------
    a, b, c : Float[Array, "3"]
        Triangle vertex positions.

    Returns
    -------
    Float[Array, "3"]
        Unit normal vector.
    """
    n = jnp.cross(b - a, c - a)
    return n / jnp.clip(jnp.linalg.norm(n), 1e-12)


def quaternion_to_rot_mat(q: Float[jax.Array, "4"]) -> Float[jax.Array, "3 3"]:
    """Convert unit quaternion to 3D rotation matrix.

    See `Quaternion rotation <https://en.wikipedia.org/wiki/Quaternions_and_spatial_rotation>`_.

    Parameters
    ----------
    q : Float[Array, "4"]
        Quaternion $[w, x, y, z]$ (normalized internally).

    Returns
    -------
    Float[Array, "3 3"]
        Rotation matrix.
    """
    a, b, c, d = q / jnp.linalg.norm(q)
    return jnp.array([[a**2+b**2-c**2-d**2, 2*b*c-2*a*d, 2*a*c+2*b*d],
                      [2*a*d+2*b*c, a**2-b**2+c**2-d**2, 2*c*d-2*a*b],
                      [2*b*d-2*a*c, 2*a*b+2*c*d, a**2-b**2-c**2+d**2]])

### Barycentric coordinates

In [40]:
#| export

def get_barycentric_coordinates(point: Float[jax.Array, " dim"],
                                a: Float[jax.Array, " dim"],
                                b: Float[jax.Array, " dim"],
                                c: Float[jax.Array, " dim"]) -> Float[jax.Array, "3"]:
    """Barycentric coordinates of point w.r.t. triangle abc.

    Uses the area-ratio method: each coordinate is the ratio of the
    sub-triangle area to the full triangle area. In 3D the point is
    projected onto the triangle plane (least-squares sense).

    Parameters
    ----------
    point : Float[Array, "dim"]
        Query point.
    a, b, c : Float[Array, "dim"]
        Triangle vertex positions.

    Returns
    -------
    Float[Array, "3"]
        Barycentric coordinates [lambda_a, lambda_b, lambda_c] summing to 1.
    """
    v0, v1, v2 = b - a, c - a, point - a
    d00, d01, d11 = jnp.dot(v0, v0), jnp.dot(v0, v1), jnp.dot(v1, v1)
    d20, d21 = jnp.dot(v2, v0), jnp.dot(v2, v1)
    denom = jnp.clip(d00 * d11 - d01 * d01, 1e-12)
    lb = (d11 * d20 - d01 * d21) / denom
    lc = (d00 * d21 - d01 * d20) / denom
    return jnp.array([1 - lb - lc, lb, lc])

In [41]:
vertices = jnp.array([[0., 0.], [0., 1.], [1., 0.]])
point1 = jnp.array([0.5, 0.2])

get_barycentric_coordinates(point1, *vertices)

Array([0.3, 0.2, 0.5], dtype=float64)

In [42]:
vertices2 = jnp.array([[0., 0., 0.], [0., 1., 0.], [1., 0., 0.]])
point2 = jnp.array([0.5, 0.2, 0.])

get_barycentric_coordinates(point2, *vertices2)

Array([0.3, 0.2, 0.5], dtype=float64)

In [43]:
point3 = jnp.array([0.5, 0.2, 1.])

get_barycentric_coordinates(point3, *vertices2)

Array([0.3, 0.2, 0.5], dtype=float64)

### Rodrigues rotation

In [44]:
#| export

def rotate_around_axis(v: Float[jax.Array, "3"],
                       axis: Float[jax.Array, "3"],
                       angle: Float[jax.Array, ""]
                       ) -> Float[jax.Array, "3"]:
    """Rotate 3D vector v by angle (radians) around unit axis using Rodrigues' formula.

    Parameters
    ----------
    v : Float[Array, "3"]
        Vector to rotate.
    axis : Float[Array, "3"]
        Unit rotation axis.
    angle : Float[Array, ""]
        Rotation angle in radians.

    Returns
    -------
    Float[Array, "3"]
        Rotated vector.
    """
    return (v * jnp.cos(angle)
            + jnp.cross(axis, v) * jnp.sin(angle)
            + axis * jnp.dot(axis, v) * (1 - jnp.cos(angle)))

In [50]:
# test: rotating x-axis by 90° around z-axis should give y-axis
x = jnp.array([1., 0., 0.])
z = jnp.array([0., 0., 1.])
result = rotate_around_axis(x, z, jnp.array(jnp.pi/2))
assert jnp.allclose(result, jnp.array([0., 1., 0.]), atol=1e-10)

# test: rotating by 0 should return the same vector
assert jnp.allclose(rotate_around_axis(x, z, jnp.array(0.)), x, atol=1e-10)

# test: rotating around the vector itself should return it
v = jnp.array([1., 2., 3.])
axis = v / jnp.linalg.norm(v)
assert jnp.allclose(rotate_around_axis(v, axis, jnp.array(1.23)), v, atol=1e-10)